# Lab 5 — Chicago Crimes: czyszczenie, optymalizacja i analiza w PySpark

**Architektura Aplikacji w Pythonie — [Lab 5] Data-intensive apps / PySpark**

Zadanie:
1. **Czyszczenie danych** — usunięcie duplikatów, braków i błędnych dat.
2. **UDF** klasyfikujący porę dnia (noc / rano / dzień / wieczór).
3. **Optymalizacja** — `cache()`, **broadcast join**, **partycjonowanie po roku**.
4. **Zapis do Parquet** z podziałem na partycje.
5. **Analiza statystyczna** przestępstw wg typu, lokalizacji i czasu + `.explain()`.
6. *(Opcjonalnie)* **MLlib** — model klasyfikujący typ przestępstwa.

> **Uwaga o środowisku:** PySpark wymaga JDK 8/11/17 oraz Pythona ≤ 3.12.
> Notebook był pisany pod takie środowisko — uruchom go tam, gdzie masz działającego Sparka
> (lokalnie w venv z `pip install pyspark`, albo na klastrze / w Colab).

## 1. Konfiguracja sesji Spark

**Teoria:** `SparkSession` to punkt wejścia do API DataFrame/SQL. Ustawiamy `JAVA_HOME`
(Spark działa na JVM) oraz `PYSPARK_PYTHON` — *workery* muszą używać tego samego interpretera
Pythona co sterownik, inaczej dostaniemy trudne do zdiagnozowania błędy serializacji.
`spark.sql.shuffle.partitions` = 8 ogranicza liczbę partycji przy *shuffle* — dla danych
rzędu dziesiątek tysięcy wierszy domyślne 200 byłoby ogromnym narzutem.

In [1]:
import os
import sys
import shutil

if "JAVA_HOME" not in os.environ:
    for candidate in [
        "/usr/lib/jvm/java-17-openjdk",                                    # Arch / CachyOS
        "/usr/lib/jvm/java-17-openjdk-amd64",                              # Debian/Ubuntu
        "/usr/lib/jvm/java-11-openjdk",
        "/usr/lib/jvm/java-11-openjdk-amd64",
        "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",  # macOS Apple Silicon
        "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",     # macOS Intel
    ]:
        if os.path.exists(candidate):
            os.environ["JAVA_HOME"] = candidate
            print(f"JAVA_HOME -> {candidate}")
            break
    else:
        print("UWAGA: nie znaleziono JDK -- ustaw JAVA_HOME recznie (JDK 8/11/17).")

# Workery Sparka musza uzywac tego samego Pythona co kernel
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf, broadcast

spark = (
    SparkSession.builder
    .appName("Chicago Crimes")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}  |  Python {sys.version.split()[0]}")

JAVA_HOME -> /usr/lib/jvm/java-17-openjdk-amd64
Spark 4.0.2  |  Python 3.12.13


## 2. Wczytanie surowych danych

**Teoria:** `read.csv` to *transformacja leniwa* — Spark zbuduje plan, ale nie ruszy danych
aż do pierwszej *akcji* (np. `count()`).

Kluczowa opcja dla tego zbioru: **`multiLine=True`**. Kolumna `location` zawiera wewnątrz
cudzysłowów znaki nowej linii (`"\n(41.78, -87.76)"`). Bez `multiLine` Spark potraktowałby
jeden rekord jako kilka i rozjechałby wszystkie kolumny. `inferSchema=False` — wszystko
wczytujemy jako string i rzutujemy świadomie w kroku czyszczenia.

In [2]:
CSV_PATH = "chicago_crimes_sample.csv"

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)      # location ma znaki nowej linii w srodku pola
    .option("quote", '"')
    .option("escape", '"')
    .csv(CSV_PATH)
)

print(f"Wczytano: {df_raw.count():,} rekordow, {len(df_raw.columns)} kolumn")
df_raw.printSchema()
df_raw.select("id", "date", "primary_type", "location_description", "arrest", "year").show(5, truncate=False)

Wczytano: 50,000 rekordow, 22 kolumn
root
 |-- id: string (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: string (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: string (nullable = true)
 |-- domestic: string (nullable = true)
 |-- beat: string (nullable = true)
 |-- district: string (nullable = true)
 |-- ward: string (nullable = true)
 |-- community_area: string (nullable = true)
 |-- fbi_code: string (nullable = true)
 |-- x_coordinate: string (nullable = true)
 |-- y_coordinate: string (nullable = true)
 |-- year: string (nullable = true)
 |-- updated_on: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- location: string (nullable = true)

+--------+-----------------------+------------+---------------------+

## 3. Czyszczenie danych

Wykonujemy po kolei:

1. **Duplikaty** — `dropDuplicates(["id"])` (klucz biznesowy rekordu).
2. **Błędne daty** — parsujemy `date` na timestamp i odrzucamy wiersze, których nie da się sparsować.
3. **Braki** — `dropna` na kolumnach kluczowych do analizy.
4. **Rzutowanie typów** — `year`, `arrest` (bool), godzina, miesiąc, dzień tygodnia.
5. **Sanity filter** — sensowny zakres lat (2001–2026, Chicago Crimes startuje od 2001).

**Teoria:** każda z tych operacji to transformacja — łączą się w jeden DAG, a Spark wykona
je dopiero przy `count()` na końcu. Liczymy „przed” i „po”, żeby zobaczyć ile rekordów odpadło.

In [3]:
before = df_raw.count()

# 3.1 Duplikaty po kluczu biznesowym
df = df_raw.dropDuplicates(["id"])
after_dedup = df.count()

# 3.2 Parsowanie daty + odrzucenie bledow (format ISO: 2026-05-07T00:00:00.000)
df = df.withColumn("date_ts", F.to_timestamp("date", "yyyy-MM-dd'T'HH:mm:ss.SSS"))
df = df.filter(F.col("date_ts").isNotNull())

# 3.3 Braki w kolumnach kluczowych
KEY_COLS = ["id", "primary_type", "location_description", "year", "arrest", "date_ts"]
df = df.dropna(subset=KEY_COLS)

# 3.4 Rzutowanie typow + kolumny pochodne czasu
df = (
    df
    .withColumn("year_int",    F.col("year").cast("int"))
    .withColumn("arrest_bool", F.col("arrest") == "true")
    .withColumn("domestic_bool", F.col("domestic") == "true")
    .withColumn("hour",         F.hour("date_ts"))
    .withColumn("month",        F.month("date_ts"))
    .withColumn("dow",          F.date_format("date_ts", "E"))   # Mon, Tue, ...
)

# 3.5 Sensowny zakres lat (Chicago Crimes: 2001+)
df = df.filter((F.col("year_int") >= 2001) & (F.col("year_int") <= 2026))

after = df.count()
print(f"Surowe:           {before:,}")
print(f"Po deduplikacji:  {after_dedup:,}  (usunieto {before - after_dedup:,} duplikatow)")
print(f"Po czyszczeniu:   {after:,}  (usunieto lacznie {before - after:,} rekordow)")

Surowe:           50,000
Po deduplikacji:  50,000  (usunieto 0 duplikatow)
Po czyszczeniu:   49,803  (usunieto lacznie 197 rekordow)


## 4. UDF — klasyfikacja pory dnia

**Teoria:** UDF (*User Defined Function*) pozwala wpleść własną logikę Pythona w pipeline
Sparka. **Uwaga:** UDF jest „czarną skrzynką” dla optymalizatora Catalyst i wymaga
serializacji danych Python↔JVM, więc jest wolniejszy niż funkcje wbudowane / `F.when()`.
Tu używamy UDF, bo wymaga tego zadanie — w produkcji tę samą logikę zapisalibyśmy przez
`F.when(...).otherwise(...)`, co pokazujemy w komentarzu.

In [4]:
@udf(StringType())
def pora_dnia(hour):
    """Mapuje godzine (0-23) na pore dnia."""
    if hour is None:
        return "nieznana"
    if 5 <= hour < 12:
        return "rano"
    elif 12 <= hour < 17:
        return "dzien"
    elif 17 <= hour < 22:
        return "wieczor"
    return "noc"          # 22:00 - 4:59

df = df.withColumn("pora_dnia", pora_dnia("hour"))

# Rownowazne bez UDF (szybsze, widoczne dla Catalyst):
# df = df.withColumn("pora_dnia",
#     F.when((F.col("hour") >= 5)  & (F.col("hour") < 12), "rano")
#      .when((F.col("hour") >= 12) & (F.col("hour") < 17), "dzien")
#      .when((F.col("hour") >= 17) & (F.col("hour") < 22), "wieczor")
#      .otherwise("noc"))

df.select("date_ts", "hour", "pora_dnia").show(8, truncate=False)

+-------------------+----+---------+
|date_ts            |hour|pora_dnia|
+-------------------+----+---------+
|2026-02-14 06:10:00|6   |rano     |
|2026-02-14 05:04:00|5   |rano     |
|2026-02-14 07:10:00|7   |rano     |
|2026-02-14 05:16:00|5   |rano     |
|2026-02-14 08:10:00|8   |rano     |
|2026-02-14 05:00:00|5   |rano     |
|2026-02-14 06:30:00|6   |rano     |
|2026-02-14 07:12:00|7   |rano     |
+-------------------+----+---------+
only showing top 8 rows


## 5. Optymalizacja: `cache()`

**Teoria:** `df` będzie podstawą wielu agregacji poniżej. Bez `cache()` Spark przeliczałby
cały pipeline (wczytanie + czyszczenie + UDF) od nowa przy **każdej** akcji. `cache()`
materializuje wynik w pamięci po pierwszej akcji i kolejne operacje czytają z pamięci.
`count()` zaraz po `cache()` wymusza materializację (cache jest *leniwy*).

In [5]:
df.cache()
n = df.count()   # materializacja cache
print(f"DataFrame zabuforowany w pamieci: {n:,} rekordow gotowych do analizy.")

DataFrame zabuforowany w pamieci: 49,803 rekordow gotowych do analizy.


## 6. Optymalizacja: broadcast join (słownik kodów FBI)

**Teoria:** Standardowy join robi *shuffle* — przesyła dane obu tabel po sieci wg klucza
(kosztowne). Gdy jedna tabela jest **mała**, lepiej ją **rozgłosić** (`broadcast`): Spark
wysyła jej kopię do każdego executora i join odbywa się lokalnie, bez shuffle dużej tabeli.
W planie zobaczymy `BroadcastHashJoin` zamiast `SortMergeJoin`. Tu mała tabela to słownik
kodów FBI (kilkadziesiąt wierszy).

In [6]:
fbi_lookup = spark.createDataFrame([
    ("01A", "Homicide"),          ("01B", "Homicide"),
    ("02",  "Criminal Sexual Assault"), ("03",  "Robbery"),
    ("04A", "Aggravated Assault"), ("04B", "Aggravated Battery"),
    ("05",  "Burglary"),          ("06",  "Larceny / Theft"),
    ("07",  "Motor Vehicle Theft"),("08A", "Simple Assault"),
    ("08B", "Simple Battery"),     ("09",  "Arson"),
    ("10",  "Forgery / Counterfeiting"), ("11",  "Fraud"),
    ("12",  "Embezzlement"),       ("13",  "Stolen Property"),
    ("14",  "Vandalism"),          ("15",  "Weapons Violation"),
    ("16",  "Prostitution"),       ("17",  "Sex Offense"),
    ("18",  "Narcotics"),          ("19",  "Gambling"),
    ("20",  "Offenses vs. Family"),("22",  "Liquor Law Violation"),
    ("24",  "Disorderly Conduct"), ("26",  "Other / Misc"),
], ["fbi_code", "fbi_category"])

df_joined = df.join(broadcast(fbi_lookup), on="fbi_code", how="left")

print("=== Plan joinu (oczekujemy BroadcastHashJoin) ===")
df_joined.explain(mode="simple")
df_joined.select("primary_type", "fbi_code", "fbi_category").show(5, truncate=False)

=== Plan joinu (oczekujemy BroadcastHashJoin) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [fbi_code#14, id#0, case_number#1, date#2, block#3, iucr#4, primary_type#5, description#6, location_description#7, arrest#8, domestic#9, beat#10, district#11, ward#12, community_area#13, x_coordinate#15, y_coordinate#16, year#17, updated_on#18, latitude#19, longitude#20, location#21, date_ts#169, year_int#170, arrest_bool#171, ... 6 more fields]
   +- BroadcastHashJoin [fbi_code#14], [fbi_code#1543], LeftOuter, BuildRight, false
      :- InMemoryTableScan [id#0, case_number#1, date#2, block#3, iucr#4, primary_type#5, description#6, location_description#7, arrest#8, domestic#9, beat#10, district#11, ward#12, community_area#13, fbi_code#14, x_coordinate#15, y_coordinate#16, year#17, updated_on#18, latitude#19, longitude#20, location#21, date_ts#169, year_int#170, arrest_bool#171, ... 5 more fields]
      :     +- InMemoryRelation [id#0, case_number#1, date#2, block#3, iucr

## 7. Zapis do Parquet z partycjonowaniem

**Teoria:** **Parquet** to kolumnowy format binarny — kompresuje dane i pozwala czytać tylko
potrzebne kolumny. **Partycjonowanie** zapisuje dane do podkatalogów wg wartości kolumny
(`year_int=2026/month=5/...`). Przy odczycie z filtrem po tej kolumnie Spark czyta **tylko
właściwe katalogi** (*partition pruning* / *predicate pushdown*) — pomija resztę danych.

Zadanie wymaga partycjonowania **po roku**; dokładamy `month` jako drugi poziom, żeby
demonstracja prunningu była wyraźna (próbka zawiera tylko rok 2026, więc sam rok dałby
jedną partycję).

In [7]:
OUT_DIR = "chicago_crimes_parquet"
if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)

(
    df.select(
        "id", "case_number", "date_ts", "primary_type", "description",
        "location_description", "arrest_bool", "domestic_bool",
        "pora_dnia", "hour", "dow", "district", "community_area",
        "fbi_code", "year_int", "month",
    )
    .write.mode("overwrite")
    .partitionBy("year_int", "month")
    .parquet(OUT_DIR)
)

print(f"Zapisano partycjonowany Parquet -> {OUT_DIR}/")
for p in sorted(os.listdir(OUT_DIR))[:5]:
    print("  ", p)

# Test partition pruning: czytamy tylko jeden miesiac
t = spark.read.parquet(OUT_DIR).filter((F.col("year_int") == 2026) & (F.col("month") == 5))
print(f"\nOdczyt year=2026, month=5: {t.count():,} rekordow")
print("=== Plan odczytu -- zwroc uwage na PartitionFilters ===")
t.explain(mode="simple")

Zapisano partycjonowany Parquet -> chicago_crimes_parquet/
   ._SUCCESS.crc
   _SUCCESS
   year_int=2026

Odczyt year=2026, month=5: 3,758 rekordow
=== Plan odczytu -- zwroc uwage na PartitionFilters ===
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id#3205,case_number#3206,date_ts#3207,primary_type#3208,description#3209,location_description#3210,arrest_bool#3211,domestic_bool#3212,pora_dnia#3213,hour#3214,dow#3215,district#3216,community_area#3217,fbi_code#3218,year_int#3219,month#3220] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/content/chicago_crimes_parquet], PartitionFilters: [isnotnull(year_int#3219), isnotnull(month#3220), (year_int#3219 = 2026), (month#3220 = 5)], PushedFilters: [], ReadSchema: struct<id:string,case_number:string,date_ts:timestamp,primary_type:string,description:string,loca...




## 8. Analiza statystyczna + `.explain()`

Każde zapytanie opisujemy planem wykonania (`.explain()`), żeby zobaczyć co zrobił
optymalizator Catalyst (skanowanie, `HashAggregate`, `Exchange`/shuffle, sortowanie).

### 8.1. Przestępstwa wg typu (top 10)

In [8]:
by_type = (
    df.groupBy("primary_type")
      .agg(
          F.count("*").alias("liczba"),
          F.round(F.avg(F.col("arrest_bool").cast("int")) * 100, 1).alias("pct_aresztowan"),
      )
      .orderBy(F.col("liczba").desc())
)
by_type.show(10, truncate=False)
print("=== explain: groupBy(primary_type) ===")
by_type.explain(mode="formatted")

+-------------------+------+--------------+
|primary_type       |liczba|pct_aresztowan|
+-------------------+------+--------------+
|THEFT              |10920 |9.1           |
|BATTERY            |9253  |18.0          |
|CRIMINAL DAMAGE    |5497  |4.6           |
|ASSAULT            |4564  |12.8          |
|MOTOR VEHICLE THEFT|3886  |3.9           |
|OTHER OFFENSE      |3459  |15.3          |
|BURGLARY           |3163  |2.7           |
|DECEPTIVE PRACTICE |2486  |2.4           |
|NARCOTICS          |1455  |93.1          |
|CRIMINAL TRESPASS  |1193  |34.5          |
+-------------------+------+--------------+
only showing top 10 rows
=== explain: groupBy(primary_type) ===
== Physical Plan ==
AdaptiveSparkPlan (32)
+- Sort (31)
   +- Exchange (30)
      +- HashAggregate (29)
         +- Exchange (28)
            +- HashAggregate (27)
               +- InMemoryTableScan (1)
                     +- InMemoryRelation (2)
                           +- AdaptiveSparkPlan (26)
                  

### 8.2. Przestępstwa wg lokalizacji (top 10)

In [9]:
by_loc = (
    df.groupBy("location_description")
      .count()
      .orderBy(F.col("count").desc())
)
by_loc.show(10, truncate=False)
print("=== explain: groupBy(location_description) ===")
by_loc.explain(mode="simple")

+--------------------------------------+-----+
|location_description                  |count|
+--------------------------------------+-----+
|STREET                                |13979|
|APARTMENT                             |9802 |
|RESIDENCE                             |5692 |
|SIDEWALK                              |2308 |
|PARKING LOT / GARAGE (NON RESIDENTIAL)|1842 |
|SMALL RETAIL STORE                    |1708 |
|DEPARTMENT STORE                      |1147 |
|RESTAURANT                            |1107 |
|ALLEY                                 |991  |
|VEHICLE NON-COMMERCIAL                |923  |
+--------------------------------------+-----+
only showing top 10 rows
=== explain: groupBy(location_description) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#4498L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(count#4498L DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=935]
      +- HashAggregate(keys=[location_description#7], functions=[c

### 8.3. Przestępstwa wg czasu — pora dnia, godzina, dzień tygodnia

In [10]:
print("=== Wg pory dnia ===")
by_time = (
    df.groupBy("pora_dnia")
      .agg(
          F.count("*").alias("liczba"),
          F.round(F.avg(F.col("arrest_bool").cast("int")) * 100, 1).alias("pct_aresztowan"),
      )
      .orderBy(F.col("liczba").desc())
)
by_time.show()

print("=== Rozklad godzinowy ===")
df.groupBy("hour").count().orderBy("hour").show(24)

print("=== Wg dnia tygodnia ===")
df.groupBy("dow").count().orderBy(F.col("count").desc()).show()

print("=== explain: groupBy(pora_dnia) ===")
by_time.explain(mode="simple")

=== Wg pory dnia ===
+---------+------+--------------+
|pora_dnia|liczba|pct_aresztowan|
+---------+------+--------------+
|    dzien| 13304|          15.7|
|  wieczor| 12778|          17.5|
|      noc| 12154|          13.7|
|     rano| 11567|          12.3|
+---------+------+--------------+

=== Rozklad godzinowy ===
+----+-----+
|hour|count|
+----+-----+
|   0| 2888|
|   1| 1466|
|   2| 1408|
|   3| 1330|
|   4| 1053|
|   5|  913|
|   6| 1040|
|   7| 1344|
|   8| 1793|
|   9| 2036|
|  10| 2171|
|  11| 2270|
|  12| 2780|
|  13| 2336|
|  14| 2486|
|  15| 2848|
|  16| 2854|
|  17| 2733|
|  18| 2728|
|  19| 2636|
|  20| 2442|
|  21| 2239|
|  22| 2149|
|  23| 1860|
+----+-----+

=== Wg dnia tygodnia ===
+---+-----+
|dow|count|
+---+-----+
|Wed| 7331|
|Sat| 7296|
|Mon| 7197|
|Tue| 7128|
|Fri| 7070|
|Sun| 7017|
|Thu| 6764|
+---+-----+

=== explain: groupBy(pora_dnia) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [liczba#5739L DESC NULLS LAST], true, 0
   +- Exchange ra

### 8.4. Analiza krzyżowa — typ × pora dnia (Window: udział % w obrębie typu)

**Teoria:** funkcja okna (`Window`) liczy agregat „obok” wiersza bez zwijania go w grupę.
Tu liczymy, jaki procent danego typu przestępstwa przypada na każdą porę dnia.

In [11]:
top5 = [r["primary_type"] for r in
        df.groupBy("primary_type").count().orderBy(F.col("count").desc()).limit(5).collect()]

w = Window.partitionBy("primary_type")
cross = (
    df.filter(F.col("primary_type").isin(top5))
      .groupBy("primary_type", "pora_dnia")
      .count()
      .withColumn("pct_typu", F.round(F.col("count") / F.sum("count").over(w) * 100, 1))
      .orderBy("primary_type", F.col("count").desc())
)
cross.show(20, truncate=False)
print("=== explain: window + groupBy ===")
cross.explain(mode="simple")

+-------------------+---------+-----+--------+
|primary_type       |pora_dnia|count|pct_typu|
+-------------------+---------+-----+--------+
|ASSAULT            |dzien    |1377 |30.2    |
|ASSAULT            |rano     |1185 |26.0    |
|ASSAULT            |wieczor  |1167 |25.6    |
|ASSAULT            |noc      |835  |18.3    |
|BATTERY            |noc      |2611 |28.2    |
|BATTERY            |wieczor  |2449 |26.5    |
|BATTERY            |dzien    |2294 |24.8    |
|BATTERY            |rano     |1899 |20.5    |
|CRIMINAL DAMAGE    |noc      |1886 |34.3    |
|CRIMINAL DAMAGE    |rano     |1292 |23.5    |
|CRIMINAL DAMAGE    |wieczor  |1277 |23.2    |
|CRIMINAL DAMAGE    |dzien    |1042 |19.0    |
|MOTOR VEHICLE THEFT|noc      |1249 |32.1    |
|MOTOR VEHICLE THEFT|wieczor  |1160 |29.9    |
|MOTOR VEHICLE THEFT|rano     |795  |20.5    |
|MOTOR VEHICLE THEFT|dzien    |682  |17.6    |
|THEFT              |dzien    |3640 |33.3    |
|THEFT              |rano     |2717 |24.9    |
|THEFT       

### 8.5. Wykorzystanie broadcast join — aresztowalność wg kategorii FBI

In [12]:
by_fbi = (
    df_joined.filter(F.col("fbi_category").isNotNull())
      .groupBy("fbi_category")
      .agg(
          F.count("*").alias("liczba"),
          F.round(F.avg(F.col("arrest_bool").cast("int")) * 100, 1).alias("pct_aresztowan"),
      )
      .orderBy(F.col("liczba").desc())
)
by_fbi.show(15, truncate=False)

+-------------------+------+--------------+
|fbi_category       |liczba|pct_aresztowan|
+-------------------+------+--------------+
|Larceny / Theft    |12960 |7.8           |
|Simple Battery     |7866  |18.1          |
|Vandalism          |5497  |4.6           |
|Other / Misc       |4065  |23.3          |
|Simple Assault     |3986  |7.9           |
|Motor Vehicle Theft|3886  |3.9           |
|Fraud              |2322  |2.3           |
|Aggravated Battery |1490  |16.6          |
|Narcotics          |1454  |93.2          |
|Aggravated Assault |1387  |19.9          |
|Burglary           |1123  |5.5           |
|Weapons Violation  |1062  |74.6          |
|Robbery            |916   |6.3           |
|Disorderly Conduct |457   |69.1          |
|Sex Offense        |395   |5.3           |
+-------------------+------+--------------+
only showing top 15 rows


## 9. (Opcjonalnie) MLlib — klasyfikacja typu przestępstwa

**Teoria:** Budujemy `Pipeline` MLlib przewidujący `primary_type` na podstawie godziny,
pory dnia, lokalizacji, dnia tygodnia, dzielnicy i flag (domestic). Etapy:

- `StringIndexer` — zamienia kategorie tekstowe na indeksy liczbowe (osobno dla cech i etykiety),
- `OneHotEncoder` — kategorie → wektory rzadkie (bez sztucznego porządku),
- `VectorAssembler` — sklejenie wszystkich cech w jeden wektor `features`,
- `RandomForestClassifier` — model.

Zbiór ma ~30 klas — to trudny problem wieloklasowy; baseline (zawsze „THEFT”) to ~22%
trafności, więc liczymy na wynik wyraźnie powyżej tego progu.

In [13]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

ml_df = df.select(
    "primary_type", "hour", "pora_dnia", "location_description",
    "dow", "district", "domestic_bool",
).dropna()

cat_cols = ["pora_dnia", "location_description", "dow", "district"]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in cat_cols
]
encoder = OneHotEncoder(
    inputCols=[c + "_idx" for c in cat_cols],
    outputCols=[c + "_oh" for c in cat_cols],
    handleInvalid="keep",
)
label_indexer = StringIndexer(inputCol="primary_type", outputCol="label", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=[c + "_oh" for c in cat_cols] + ["hour", "domestic_bool"],
    outputCol="features",
)
rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                            numTrees=60, maxDepth=12, maxBins=200, seed=42)

pipeline = Pipeline(stages=indexers + [encoder, label_indexer, assembler, rf])

In [14]:
train, test = ml_df.randomSplit([0.8, 0.2], seed=42)
train.cache(); test.cache()
print(f"Train: {train.count():,}  |  Test: {test.count():,}")

model = pipeline.fit(train)
pred = model.transform(test)

acc = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(pred)
f1 = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1").evaluate(pred)

print(f"\nAccuracy: {acc:.3f}")
print(f"F1-score: {f1:.3f}")

# Baseline: klasa wiekszosciowa
maj = test.groupBy("primary_type").count().orderBy(F.col("count").desc()).first()
print(f"Baseline (zawsze '{maj['primary_type']}'): {maj['count']/test.count():.3f}")

pred.select("primary_type", "prediction", "probability").show(8, truncate=False)

Train: 40,002  |  Test: 9,801

Accuracy: 0.336
F1-score: 0.232
Baseline (zawsze 'THEFT'): 0.223
+------------+----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|primary_type|prediction|probability                                                                                                                                                                                                                

### 9.1. Ważność cech

In [15]:
rf_model = model.stages[-1]
importances = rf_model.featureImportances
# Top cechy wg waznosci (na poziomie surowych kolumn -- agregat dla onehot pomijamy dla zwiezlosci)
print("Liczba cech w wektorze:", importances.size)
top = sorted(enumerate(importances.toArray()), key=lambda x: -x[1])[:10]
print("Top 10 indeksow cech wg waznosci:")
for idx, imp in top:
    print(f"  feature[{idx}]: {imp:.4f}")

Liczba cech w wektorze: 147
Top 10 indeksow cech wg waznosci:
  feature[146]: 0.3005
  feature[5]: 0.1060
  feature[10]: 0.0725
  feature[11]: 0.0701
  feature[6]: 0.0471
  feature[145]: 0.0440
  feature[8]: 0.0359
  feature[17]: 0.0243
  feature[7]: 0.0226
  feature[2]: 0.0214


## 10. Sprzątanie

In [16]:
df.unpersist()
spark.stop()
print("Sesja Spark zamknieta. Analiza zakonczona.")

Sesja Spark zamknieta. Analiza zakonczona.
